In [1]:
#| default_exp frida

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.10/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")

In [5]:
model_path = 'fred'

In [6]:
#| export
from optimum.onnxruntime import ORTModelForSeq2SeqLM

In [7]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration, AutoTokenizer

In [8]:
def convert_and_save_model(model_path, save_dir):
    model = ORTModelForSeq2SeqLM.from_pretrained(model_path, export=True)
    model.save_pretrained(save_dir)
    
    # Also save the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to {save_dir}")

In [9]:
#convert_and_save_model(full_path, full_path+'/optimized')

In [10]:
#| export
tokenizer = GPT2Tokenizer.from_pretrained(full_path+'/optimized/', eos_token='</s>')
model = ORTModelForSeq2SeqLM.from_pretrained(full_path+'/optimized/', provider="CUDAExecutionProvider")

2024-10-01 15:48:46.383981193 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-10-01 15:48:46.384001782 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-10-01 15:48:47.038492496 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-10-01 15:48:47.038510175 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-10-01 15:48:47.905908088 [W:onnxrun

In [11]:
tokenizer.model_max_length

1000000000000000019884624838656

In [12]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

In [13]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.1):
    max_input = 2*1024 - length
    prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')

    blocked_tokens = ["».",'.\n','\n\t\t','http://',',[','("','.]',' («',')','\u2004',']','(«','[', ' [', '(', ' (', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    bad_words_ids = iftoken(tokenizer, blocked_tokens)
    bad_words_ids = [[w] for w in bad_words_ids]
    
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    torch.cuda.empty_cache()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, bad_words_ids = bad_words_ids,
                        num_return_sequences=num_samples,)
    
    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [14]:
length = 50

In [15]:
max_input = 1024*2 - length
prompt = '<LM>'+'На словах ты Лев Толстой, а на деле'*100000
prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')


In [16]:
%%time
get_sample(prompt, length, 4, False)

CPU times: user 1.6 s, sys: 230 ms, total: 1.83 s
Wall time: 1.22 s


[', а на делеНа словах ты Лев Толстой… На этих строках я остановился. Я не знал точно – что именно сказать дальше и как это сделать правильно?',
 ', а на делеНа словах ты Лев Толстой. А потом я понял: это не так уж и важно – кто он такой по жизни? Важно то же самое знать о себе самом… И вот тогда-то мне стало ясно все остальное!',
 ', а на делеНа словах ты Лев Толстой… На этих строках я остановился. Я не знал точно – что именно сказать дальше и как это сделать правильно? И решил просто повторить слова Толстого: «Ты есть то же самое».',
 ', а на делеНа словах ты Лев Толстой… На этих строках я остановился. Я не знал точно – что именно хотел сказать автор этой фразы? Что он имел в виду под словами «на делах»?']

In [17]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.16 s, sys: 84.5 ms, total: 1.24 s
Wall time: 669 ms


[' – просто говно. И не надо мне тут про «венец творения». Я знаю, что ты такое на самом деле… Ты даже хуже чем дерьмо! Но я тебя люблю и уважаю за то хорошее дело которое делаешь для человечества!',
 ' – просто говно. И не надо мне тут про «свободу слова». Я сам знаю, что это такое и как оно работает… А вот ты попробуй-ка объяснить человеку с улицы: почему он должен верить в то же самое дерьмо?',
 ' – просто говно. И не надо мне тут про «любовь к ближнему». Я тебя знаю, ты и сам знаешь… Ты же врешь все время!',
 ' – просто говно. И не надо мне тут про «венец творения». Я знаю, что ты такое на самом деле… Ты даже хуже чем дерьмо!']

In [18]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.17 s, sys: 0 ns, total: 1.17 s
Wall time: 582 ms


[' – говно. И не надо мне тут про «свободу слова».',
 ' – просто говно. И не надо мне тут про «свободу слова». Я знаю, что такое свобода и как она выглядит на практике… Ты думаешь о том же самом? О свободе говорить правду в глаза?',
 ' – просто говно. И не надо мне тут про «свободу слова». Я сам знаю, что это такое и как оно работает… А вот ты попробуй объяснить человеку с улицы: почему он должен верить в то же самое дерьмо?',
 ' – просто говно. И не надо мне тут про «свободу». Я сам знаю, что такое свобода и как она выглядит на самом деле… А ты лучше скажи: почему у тебя в руках эта книга? Ты же знаешь все ответы!']

In [19]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.15 s, sys: 18.8 ms, total: 1.17 s
Wall time: 581 ms


[' – говно. И не надо мне тут про «свободу».',
 ' – просто говно. И не надо мне тут про «свободу слова». Я сам знаю, что это такое и как оно работает… А ты лучше скажи: почему у тебя в кабинете висит портрет Сталина? Ты его любишь или ненавидишь?',
 ' – просто говно. И не надо мне тут про «свободу слова». Я сам знаю, что это такое и как оно работает… А вот ты попробуй объяснить человеку с улицы: почему он должен верить в то же самое дерьмо?',
 ' – просто говно. И не надо мне тут про «свободу». Я сам знаю, что такое свобода и как она выглядит на самом деле… А ты лучше скажи: почему у тебя в голове столько мусора? Ты же умный человек!']